In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, multiprocessing, warnings
from tqdm import tqdm

import torch
import torch.nn as nn 
import torch.optim as optim
from torchvision import transforms, datasets, models
from torch.utils.data import TensorDataset, DataLoader, random_split
from  PIL import Image 

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
torch.manual_seed(42)

In [4]:
train = datasets.ImageFolder(r'data\train')
test = datasets.ImageFolder(r'data\test')

In [5]:
train.class_to_idx

{'angry': 0,
 'disgust': 1,
 'fear': 2,
 'happy': 3,
 'neutral': 4,
 'sad': 5,
 'surprise': 6}

In [6]:
train_transform = transforms.Compose([
    transforms.Resize(48),
    transforms.RandomHorizontalFlip(),
    transforms.RandomGrayscale(),
    transforms.RandomRotation(5),

    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.Resize(48),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [7]:
class datasetsMaker(torch.utils.data.Dataset):
    def __init__(self, data, transf):
        self.data = data
        self.transf = transf

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        image = self.data[index][0]
        label = self.data[index][1]
        return self.transf(image), label

In [8]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_lyr = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Dropout2d(0.2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2),  
            nn.Dropout2d(0.3),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2,2),  # may need to remove for improvement
            nn.Dropout2d(0.4)                                                                                                                         
        )

        self.linear_lyr = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(512, 7)
        )
    def forward(self, x):
        x = self.conv_lyr(x)
        x = self.linear_lyr(x)
        return x

In [9]:
training_set = datasetsMaker(train, train_transform)
validation_set = datasetsMaker(test, test_transform)

In [10]:
train_loader = DataLoader(training_set, batch_size=64, shuffle=True)
val_loader = DataLoader(validation_set, batch_size=64)

In [11]:
model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

In [22]:
epochs = 50
best_acc = 0.0

for epoch in range(epochs):
    running_loss = 0.0
    total = 0
    correct = 0


# training ---
    model.train()
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]', colour='green'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    batch_loss = running_loss / len(train_loader)
    scheduler.step(batch_loss)

# validation ---
    model.eval()
    with torch.no_grad():
        

        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]', colour='green'):
            images, labels = images.to(device), labels.to(device)

            output = model(images)
            pred = output.argmax(dim=1)
            total += len(labels)
            correct += (pred == labels).sum().item()
        val_accuracy = correct / total * 100

        if val_accuracy > best_acc:
            best_acc = val_accuracy
            torch.save(model.state_dict(), 'best_parameters.pth')

    print('---'*15)
    print(f'For epoch - {epoch+1}, training loss is - {batch_loss:.4f}, accuracy is {val_accuracy:.4f}')
    print('---'*15)


Epoch 1/50 [Val]: 100%|██████████| 113/113 [00:07<00:00, 15.57it/s]


---------------------------------------------
For epoch - 1, training loss is - 1.6758, accuracy is 45.6812
---------------------------------------------


Epoch 2/50 [Val]: 100%|██████████| 113/113 [00:04<00:00, 23.23it/s]


---------------------------------------------
For epoch - 2, training loss is - 1.3744, accuracy is 51.0309
---------------------------------------------


Epoch 3/50 [Val]: 100%|██████████| 113/113 [00:04<00:00, 22.96it/s]


...

---------------------------------------------
For epoch - 48, training loss is - 0.5372, accuracy is 67.6512
---------------------------------------------


Epoch 49/50 [Val]: 100%|██████████| 113/113 [00:05<00:00, 22.29it/s]


---------------------------------------------
For epoch - 49, training loss is - 0.5408, accuracy is 67.6651
---------------------------------------------


Epoch 50/50 [Val]: 100%|██████████| 113/113 [00:04<00:00, 23.57it/s]


---------------------------------------------
For epoch - 50, training loss is - 0.5281, accuracy is 67.8323
---------------------------------------------


In [23]:
model.load_state_dict(torch.load('best_parameters.pth'))

<All keys matched successfully>

In [12]:
best_acc

NameError: name 'best_acc' is not defined

## Using ResNet-18 as Transfer Learning

In [13]:
import torchvision.models as models

In [14]:
def resnet_transform(is_train: bool):
    if is_train:
        return transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.Grayscale(3),
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])
    else:
        return transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.Grayscale(3),
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])

In [15]:
training_set_R18 = datasetsMaker(train, resnet_transform(True))
validation_set_R18 = datasetsMaker(test, resnet_transform(False))

In [16]:
train_loader_R18 = DataLoader(training_set_R18, batch_size=64, shuffle=True)
val_loader_R18 = DataLoader(validation_set_R18, batch_size=64)

In [27]:
class CNN_ResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # for pram in self.model.parameters():
        #     pram.requires_grad = False

        in_feature = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Linear(in_feature, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        return self.model(x)

In [28]:
CNN_ResNet().parameters

<bound method Module.parameters of CNN_ResNet(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, 

In [ ]:
model = CNN_ResNet().to(device)
optimizer = optim.Adam(model.parameters(), lr = 0.001)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
    )

In [30]:

epochs = 30
best_acc = 0.0

for epoch in range(epochs):
    running_loss = 0.0
    total = 0
    correct = 0

    # training ---
    model.train()
    for images, labels in tqdm(train_loader_R18, desc=f'Epoch {epoch+1}/{epochs} [Train]', colour='green'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    batch_loss = running_loss / len(train_loader_R18)
    scheduler.step(batch_loss)  # ✅ pass loss to scheduler

    # validation ---
    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(val_loader_R18, desc=f'Epoch {epoch+1}/{epochs} [Val]', colour='green'):
            images, labels = images.to(device), labels.to(device)

            output = model(images)
            pred = output.argmax(dim=1)
            total += len(labels)
            correct += (pred == labels).sum().item()

        val_accuracy = correct / total * 100

        if val_accuracy > best_acc:
            best_acc = val_accuracy
            torch.save(model.state_dict(), 'best_parameters.pth')

    print('---' * 15)
    print(f'Epoch {epoch+1} | Loss: {batch_loss:.4f} | Val Acc: {val_accuracy:.4f}% | LR: {optimizer.param_groups[0]["lr"]:.6f}')
    print('---' * 15)

Epoch 1/30 [Val]: 100%|██████████| 113/113 [01:10<00:00,  1.60it/s]


---------------------------------------------
Epoch 1 | Loss: 1.2759 | Val Acc: 55.9905% | LR: 0.001000
---------------------------------------------


Epoch 2/30 [Val]: 100%|██████████| 113/113 [00:16<00:00,  7.02it/s]


---------------------------------------------
Epoch 2 | Loss: 1.0820 | Val Acc: 59.3341% | LR: 0.001000
---------------------------------------------


Epoch 3/30 [Val]: 100%|██████████| 113/113 [00:15<00:00,  7.45it/s]


...

---------------------------------------------
Epoch 28 | Loss: 0.0503 | Val Acc: 63.1931% | LR: 0.001000
---------------------------------------------


Epoch 29/30 [Val]: 100%|██████████| 113/113 [01:20<00:00,  1.40it/s]


---------------------------------------------
Epoch 29 | Loss: 0.0454 | Val Acc: 62.6219% | LR: 0.001000
---------------------------------------------


Epoch 30/30 [Val]: 100%|██████████| 113/113 [01:10<00:00,  1.60it/s]


---------------------------------------------
Epoch 30 | Loss: 0.0543 | Val Acc: 63.9315% | LR: 0.001000
---------------------------------------------


In [31]:
best_acc

63.931457230426304